# 4. Classification and GIS export — the full loop

The closing Stage-2 workflow: engineer per-point features, train a random forest on the AHN labels, measure accuracy **honestly** with spatially-blocked cross-validation, then export a classified GeoPackage a GIS analyst can open.

```bash
pip install "geoai3d[laz,gis]"
```

In [ ]:
import numpy as np
from geoai3d import read_lidar, geometric_features, ground, to_dtm

cloud = read_lidar("../data/ahn_sample.laz")
features = geometric_features(cloud, radius=1.0)

## A feature that matters: height above ground

Geometry alone struggles to tell a flat roof from flat ground. Height above the bare earth is the feature that separates them, so we build a DTM and subtract it at each point.

In [ ]:
classified = ground(cloud, cloth_resolution=1.0)
dtm = to_dtm(classified, resolution=1.0)

a, _, c, _, e, f = dtm.transform  # affine: x = a*col + c, y = e*row + f
col = np.clip(((cloud.xyz[:, 0] - c) / a).astype(int), 0, dtm.width - 1)
row = np.clip(((cloud.xyz[:, 1] - f) / e).astype(int), 0, dtm.height - 1)
ground_z = dtm.data[row, col]
height = cloud.xyz[:, 2] - ground_z
height = np.where(np.isnan(height), 0.0, height)
features = features.with_attribute("height_above_ground", height)

## An honest accuracy estimate

Points close together are highly correlated, so a random train/test split lets the model peek at neighbours of its test points and inflates accuracy. `spatial_block_split` assigns whole spatial blocks to folds, so test points are separated from training points. This is the number you should report.

In [ ]:
from geoai3d import (
    spatial_block_split,
    train_classifier,
    classify,
    evaluate,
)

FEATURES = ["planarity", "linearity", "sphericity", "verticality",
            "height_above_ground"]

# Each block is tested once, so pooling the out-of-fold predictions
# gives one honest, spatially-clean prediction per point.
truths, predictions = [], []
for train_idx, test_idx in spatial_block_split(
    features, block_size=25.0, n_folds=5
):
    model = train_classifier(
        features[train_idx],
        label_attribute="classification",
        feature_names=FEATURES,
    )
    predicted = classify(features[test_idx], model=model)
    truths.append(features[test_idx].attribute("classification"))
    predictions.append(predicted.attribute("prediction"))

report = evaluate(np.concatenate(truths), np.concatenate(predictions))
print(f"blocked-CV overall accuracy: {report.overall_accuracy:.3f}")
print(f"blocked-CV mean IoU        : {report.mean_iou:.3f}")

## Per-class: where the model is strong

Overall accuracy can hide a weak rare class. The per-class IoU makes it explicit: ground and building classify well, while the ~0.1% water class has far too few points to learn — which is exactly why mean IoU sits below overall accuracy. This is why we report both.

In [ ]:
names = {1: "vegetation", 2: "ground", 6: "building", 9: "water"}
for label in report.labels:
    metrics = report.per_class[label]
    print(f"{names.get(label, label):>10}: "
          f"IoU {metrics['iou']:.2f}  F1 {metrics['f1']:.2f}  "
          f"n={int(metrics['support']):,}")

## Train on everything and export

With the workflow validated, train on all the points, classify, and write a GeoPackage. The CRS goes into the file and the provenance record is written to a sidecar, so the output is fully traceable.

In [ ]:
from geoai3d import to_geopackage

model = train_classifier(
    features, label_attribute="classification", feature_names=FEATURES
)
result = classify(features, model=model)
to_geopackage(result, "classified.gpkg", attributes=["prediction", "classification", "height_above_ground"])
print("wrote classified.gpkg (+ classified.gpkg.provenance.json)")

## See the result

The classified cloud in 3D, coloured by predicted class, and the confusion matrix from the blocked-CV predictions.

In [ ]:
from geoai3d import view

view(result, color_by="prediction")

In [ ]:
import plotly.express as px

ticks = [names.get(label, str(label)) for label in report.labels]
px.imshow(
    report.confusion,
    x=ticks,
    y=ticks,
    text_auto=True,
    color_continuous_scale="Blues",
    labels={"x": "predicted", "y": "true", "color": "points"},
    title="Confusion matrix (blocked cross-validation)",
)

That is the whole Stage-2 promise: a raw LiDAR tile in, a classified, georeferenced, provenanced GIS layer out — CPU-only, in a handful of lines.